In [6]:
# ==========================================
# CELDA 1: Configuración y Carga de Infraestructura
# ==========================================
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from ortools.sat.python import cp_model
import warnings
warnings.filterwarnings('ignore')

# 1. Conectar a la BD (Ajusta tus credenciales si es necesario)
user = "Joasro"
password = "Akriila123." 
host = "localhost"
db = "dss_academico_unah"

engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{db}")

print("Cargando logística desde la BD...")

# 🛑 MAGIA: Solo traemos a los docentes habilitados para este periodo
df_docentes = pd.read_sql("SELECT * FROM docentes_activos WHERE Disponible = 1", engine)

df_docente_area = pd.read_sql("SELECT * FROM docente_area", engine)
df_aulas = pd.read_sql("SELECT * FROM espacios_fisicos", engine)

df_malla = pd.read_sql("""
    SELECT ID_Clase, Nombre_Clase, Codigo_Oficial, ID_Area, 
           Requiere_Laboratorio, Permite_Presencial, Permite_Virtual 
    FROM malla_curricular
""", engine)

df_demanda = pd.read_csv('../data/demanda_proyectada_2026.csv')
if 'ID_Clase' not in df_demanda.columns:
    df_demanda = pd.merge(df_demanda, df_malla[['Codigo_Oficial', 'ID_Clase']], on='Codigo_Oficial', how='left')

HORARIOS = [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20] 
CAPACIDAD_ESTANDAR = 25 

print(f"✅ Datos listos: {len(df_docentes)} docentes DISPONIBLES y {len(df_aulas)} aulas físicas.")

Cargando logística desde la BD...
✅ Datos listos: 8 docentes DISPONIBLES y 8 aulas físicas.


In [7]:
# ==========================================
# CELDA 2: Generador de Variables (Tolerancia de Sobrecupo 20%)
# ==========================================
import itertools
import numpy as np

model = cp_model.CpModel()
secciones_posibles = {}
duraciones = {} 
capacidades_fisicas = {} # Guardará la Capacidad EXTENDIDA (+20%)
ID_AULA_VIRTUAL = 999 
CAPACIDAD_VIRTUAL = 60 

CLASE_DEBUG = "Ninguna" 

def extraer_hora_entera(val):
    if pd.isnull(val): return None
    if hasattr(val, 'seconds'): return val.seconds // 3600
    try: return int(str(val).split(':')[0])
    except: return None

def obtener_lista_bloqueos(val):
    if pd.isnull(val) or str(val).strip() == "": return []
    try: return [int(x.strip()) for x in str(val).split(',') if x.strip().isdigit()]
    except: return []

def generar_patrones_dias(uv, dias_trabajo_docente):
    dias_profe = [d.strip() for d in str(dias_trabajo_docente).split(',') if d.strip()]
    patrones_validos = []
    if not dias_profe: return []

    if uv >= 5 and all(dia in dias_profe for dia in ['Lu','Ma','Mi','Ju','Vi']):
        patrones_validos.append('Lu,Ma,Mi,Ju,Vi')
    elif uv == 4 and all(dia in dias_profe for dia in ['Lu','Ma','Mi','Ju']):
        patrones_validos.append('Lu,Ma,Mi,Ju') 
    elif uv <= 3:
        if all(dia in dias_profe for dia in ['Lu','Mi','Vi']): patrones_validos.append('Lu,Mi,Vi')
        if all(dia in dias_profe for dia in ['Ma','Ju']): patrones_validos.append('Ma,Ju')
            
    for d in dias_profe:
        if d in ['Vi', 'Sa']: patrones_validos.append(d) 
        
    if len(dias_profe) <= 3:
        patrones_validos.append(",".join(dias_profe))
    
    return list(set(patrones_validos)) 

print("Calculando aforos con tolerancia del 20% (Sobrecupo)...")

for idx, row in df_demanda.iterrows():
    id_clase = row['ID_Clase']
    num_s_max = int(np.ceil(row['Cupos_Estimados'] / 15)) 
    if num_s_max == 0: num_s_max = 1
    
    info_c = df_malla[df_malla['ID_Clase'] == id_clase].iloc[0]
    uv = int(info_c.get('Unidades_Valorativas', 4))
    area_clase = int(info_c['ID_Area']) 
    codigo_clase = info_c['Codigo_Oficial']
    p_presencial, p_virtual = info_c['Permite_Presencial'], info_c['Permite_Virtual']

    for s in range(num_s_max):
        for d_idx, doc in df_docentes.iterrows():
            id_doc, tipo_d = doc['ID_Docente'], doc['Tipo_Docente']
            
            especialidades = [int(x) for x in df_docente_area[df_docente_area['ID_Docente'] == id_doc]['ID_Area'].values]
            if area_clase not in especialidades: continue

            patrones_dias = generar_patrones_dias(uv, doc['Dias_Trabajo'])
            if not patrones_dias: continue 

            h_inicio = extraer_hora_entera(doc['Hora_Inicio_Turno'])
            h_fin = extraer_hora_entera(doc['Hora_Fin_Turno'])
            bloqueos = obtener_lista_bloqueos(doc['Horas_Bloqueadas']) 

            for patron in patrones_dias:
                cantidad_dias = len(patron.split(','))
                duracion = int(np.ceil(uv / cantidad_dias)) 
                
                for h in HORARIOS:
                    if h_inicio is not None and h < h_inicio: continue
                    if h_fin is not None and (h + duracion) > h_fin: continue
                    if any(hora_clase in bloqueos for hora_clase in range(h, h + duracion)): continue 
                    
                    if tipo_d == 'Tegucigalpa' and p_virtual == 1:
                        v_key = (id_clase, s, id_doc, patron, h, ID_AULA_VIRTUAL, 'Virtual')
                        secciones_posibles[v_key] = model.NewBoolVar(f"c{id_clase}_s{s}_d{id_doc}_p{patron}_h{h}_V")
                        duraciones[v_key] = duracion
                        capacidades_fisicas[v_key] = CAPACIDAD_VIRTUAL
                    else:
                        if p_virtual == 1:
                            v_key = (id_clase, s, id_doc, patron, h, ID_AULA_VIRTUAL, 'Virtual')
                            secciones_posibles[v_key] = model.NewBoolVar(f"c{id_clase}_s{s}_d{id_doc}_p{patron}_h{h}_V")
                            duraciones[v_key] = duracion
                            capacidades_fisicas[v_key] = CAPACIDAD_VIRTUAL
                        if p_presencial == 1:
                            for a_idx, aula in df_aulas.iterrows():
                                cap_aula_base = int(aula.get('Cupos_Maximos', aula.get('Capacidad', 30)))
                                
                                # 🛑 MAGIA DEL SOBRECUPO: Extendemos la capacidad un 20%
                                cap_aula_extendida = int(np.floor(cap_aula_base * 1.20))
                                
                                v_key = (id_clase, s, id_doc, patron, h, aula['ID_Espacio'], 'Presencial')
                                secciones_posibles[v_key] = model.NewBoolVar(f"c{id_clase}_s{s}_d{id_doc}_p{patron}_h{h}_a{aula['ID_Espacio']}")
                                duraciones[v_key] = duracion
                                capacidades_fisicas[v_key] = cap_aula_extendida # La IA trabajará con la capacidad extendida

print(f"🧩 Variables con aforo extendido generadas: {len(secciones_posibles)}")

Calculando aforos con tolerancia del 20% (Sobrecupo)...
🧩 Variables con aforo extendido generadas: 8271


In [8]:
# ==========================================
# CELDA 3: Restricciones (Choques y Candado Anti-Desperdicio)
# ==========================================
dias_semana_evaluar = ['Lu', 'Ma', 'Mi', 'Ju', 'Vi', 'Sa', 'Do']

model.Add(sum(secciones_posibles.values()) >= 15)
model.Add(sum(secciones_posibles.values()) <= 40)

# 1. Choques de Aula 
for id_a in df_aulas['ID_Espacio']:
    for h_eval in HORARIOS:
        for dia in dias_semana_evaluar:
            v_a = []
            for v_key, var in secciones_posibles.items():
                c, s, d, p, h_inicio, aula, m = v_key
                if aula == id_a and dia in p.split(',') and m != 'Virtual':
                    if h_inicio <= h_eval < (h_inicio + duraciones[v_key]):
                        v_a.append(var)
            if v_a: model.Add(sum(v_a) <= 1)

# 2. Choques de Docente y Carga Máxima
docentes_activos_vars = {}
for id_d in df_docentes['ID_Docente']:
    for h_eval in HORARIOS:
        for dia in dias_semana_evaluar:
            vars_doc = []
            for v_key, var in secciones_posibles.items():
                c, s, d, p, h_inicio, aula, m = v_key
                if d == id_d and dia in p.split(','):
                    if h_inicio <= h_eval < (h_inicio + duraciones[v_key]):
                        vars_doc.append(var)
            if vars_doc: model.Add(sum(vars_doc) <= 1)
    
    clases_profe = [var for v_key, var in secciones_posibles.items() if v_key[2] == id_d]
    if clases_profe:
        model.Add(sum(clases_profe) <= 4) 
        es_usado = model.NewBoolVar(f'u_{id_d}')
        model.Add(sum(clases_profe) > 0).OnlyEnforceIf(es_usado)
        model.Add(sum(clases_profe) == 0).OnlyEnforceIf(es_usado.Not())
        docentes_activos_vars[id_d] = es_usado

# 3. Restricción de Sección Única
for idx, row in df_demanda.iterrows():
    id_c = row['ID_Clase']
    num_s_max = int(np.ceil(row['Cupos_Estimados'] / 15))
    if num_s_max == 0: num_s_max = 1
    for sec_idx in range(num_s_max):
        ops = [var for v_key, var in secciones_posibles.items() if v_key[0] == id_c and v_key[1] == sec_idx]
        if ops: model.Add(sum(ops) <= 1)

# 🛑 4. LÍMITE FÍSICO ANTI-DESPERDICIO (ROBUSTO)
for idx, row in df_demanda.iterrows():
    id_c = row['ID_Clase']
    demanda_real = int(row['Cupos_Estimados'])
    
    sillas_asignadas = [var * capacidades_fisicas[v_key] for v_key, var in secciones_posibles.items() if v_key[0] == id_c]
    
    if sillas_asignadas:
        # Extrae el tamaño del aula más grande para dar margen de maniobra matemático seguro
        max_aula_posible = max([capacidades_fisicas[v_key] for v_key, var in secciones_posibles.items() if v_key[0] == id_c] + [40])
        # La IA no puede abrir secciones si el excedente supera la capacidad del aula más grande (Evita abrir secciones vacías)
        model.Add(sum(sillas_asignadas) <= demanda_real + max_aula_posible)

# 5. Presupuesto
def contar_contratos(tipo):
    ids = df_docentes[df_docentes['Tipo_Docente'] == tipo]['ID_Docente'].values
    return [docentes_activos_vars[i] for i in ids if i in docentes_activos_vars]

emergentes, tgu = contar_contratos('Emergente'), contar_contratos('Tegucigalpa')
if emergentes: model.Add(sum(emergentes) <= 2)
if tgu: model.Add(sum(tgu) <= 2)

print("⚖️ Restricciones Físicas (Con margen dinámico) aplicadas.")

⚖️ Restricciones Físicas (Con margen dinámico) aplicadas.


In [9]:
# ==========================================
# CELDA 4: Función Objetivo (Maximizar Sillas Extendidas)
# ==========================================
objetivo = []
for v_key, var in secciones_posibles.items():
    id_c, s, id_d, patron, h, id_a, mod = v_key
    puntos = 0
    
    tipo_d = df_docentes[df_docentes['ID_Docente'] == id_d]['Tipo_Docente'].values[0]
    nombre_clase = df_malla[df_malla['ID_Clase'] == id_c]['Nombre_Clase'].values[0].lower()
    
    capacidad_aula_extendida = capacidades_fisicas[v_key]
    puntos += capacidad_aula_extendida * 50  
    
    if any(k in nombre_clase for k in ['seminario', 'proyecto', 'práctica', 'practica', 'tesis']):
        puntos += 50000  
    elif any(k in nombre_clase for k in ['redes', 'sistemas operativos', 'auditoría', 'software']):
        puntos += 500    
    else:
        puntos += 10     
        
    if tipo_d == 'Base': puntos += 20          
    elif tipo_d == 'Tegucigalpa': puntos += 5           
    else: puntos += 1           
    
    if mod == 'Virtual':
        if h >= 17: puntos += 10      
        elif h < 12: puntos -= 5       
    else:
        necesita_lab = df_malla[df_malla['ID_Clase'] == id_c]['Requiere_Laboratorio'].values[0]
        es_lab = 1 if df_aulas[df_aulas['ID_Espacio'] == id_a]['Tipo_Espacio'].values[0].lower() == 'laboratorio' else 0
        if necesita_lab == es_lab: puntos += 5 
        
    if patron == 'Lu,Ma,Mi,Ju': puntos += 5 

    objetivo.append(var * puntos)

model.Maximize(sum(objetivo))

solver = cp_model.CpSolver()
status = solver.Solve(model)
if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print(f"✅ ¡Horario generado exitosamente! (Puntuación: {solver.ObjectiveValue()})")
else:
    print("❌ No se encontró solución.")

✅ ¡Horario generado exitosamente! (Puntuación: 121069.0)


In [10]:
# ==========================================
# CELDA 5: Calendario Maestro (Análisis Físico y Sobrecupo)
# ==========================================
import pandas as pd

if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    horario_flat = []
    cupos_cubiertos_por_clase = {} 
    
    for v_key, var in secciones_posibles.items():
        if solver.Value(var) == 1: 
            id_c, s, id_d, patron, h_inicio, id_a, mod = v_key
            
            codigo_clase = df_malla[df_malla['ID_Clase'] == id_c]['Codigo_Oficial'].values[0]
            clase = df_malla[df_malla['ID_Clase'] == id_c]['Nombre_Clase'].values[0]
            docente = df_docentes[df_docentes['ID_Docente'] == id_d]['Nombre'].values[0]
            aula_nombre = 'Virtual' if mod == 'Virtual' else df_aulas[df_aulas['ID_Espacio'] == id_a]['Nombre_Espacio'].values[0]
            
            duracion = duraciones[v_key]
            capacidad_extendida = capacidades_fisicas[v_key]
            
            # Recalculamos la base para mostrarla visualmente
            capacidad_base = int(np.ceil(capacidad_extendida / 1.20)) if mod != 'Virtual' else CAPACIDAD_VIRTUAL
            
            # Sumamos al acumulador la capacidad que incluye el 20% de tolerancia
            cupos_cubiertos_por_clase[id_c] = cupos_cubiertos_por_clase.get(id_c, 0) + capacidad_extendida
            
            dias_asignados = patron.split(',')
            for dia in dias_asignados:
                for hr_bloque in range(h_inicio, h_inicio + duracion):
                    horario_flat.append({
                        'Hora': f"{hr_bloque:02d}:00", 
                        'Dia': dia,
                        # 🛑 VISTA DIRECTIVA: Muestra la base y el sobrecupo tolerado
                        'Detalle': f"📚 {codigo_clase} - {clase} (S-{s+1})\n👨‍🏫 {docente}\n🏫 {aula_nombre} ({capacidad_base} norm / {capacidad_extendida} ext)"
                    })

    df_flat = pd.DataFrame(horario_flat)
    df_calendario = df_flat.groupby(['Hora', 'Dia'])['Detalle'].apply(lambda x: "\n\n".join(x)).reset_index()
    
    matriz_semanal = df_calendario.pivot(index='Hora', columns='Dia', values='Detalle').fillna("---")
    
    orden_dias = ['Lu', 'Ma', 'Mi', 'Ju', 'Vi', 'Sa', 'Do']
    columnas_presentes = [d for d in orden_dias if d in matriz_semanal.columns]
    matriz_semanal = matriz_semanal[columnas_presentes]

    print("🎓 CALENDARIO MAESTRO SEMANAL COMPLETADO:")
    display(matriz_semanal.style.set_properties(**{'white-space': 'pre-wrap', 'vertical-align': 'top', 'border': '1px solid black', 'padding': '10px'}))
    
    print("\n" + "="*80)
    print("📊 REPORTE DE GESTIÓN (DÉFICIT FÍSICO Y TOLERANCIAS)")
    print("="*80)
    
    alertas = 0
    for idx, row in df_demanda.iterrows():
        id_c = row['ID_Clase']
        cupos_necesarios = row['Cupos_Estimados']
        # Los logrados ya incluyen el 20% de tolerancia de cada aula asignada
        cupos_logrados = cupos_cubiertos_por_clase.get(id_c, 0) 
        
        if cupos_necesarios > cupos_logrados:
            alertas += 1
            clase_nombre = row['Nombre_Clase']
            codigo_oficial = df_malla[df_malla['ID_Clase'] == id_c]['Codigo_Oficial'].values[0]
            area_id = int(df_malla[df_malla['ID_Clase'] == id_c]['ID_Area'].values[0])
            area_nombre = pd.read_sql(f"SELECT Nombre_Area FROM areas_academicas WHERE ID_Area = {area_id}", engine).values[0][0]
            
            print(f"⚠️ DÉFICIT EN: {codigo_oficial} - {clase_nombre}")
            print(f"   - Demanda Proyectada: {cupos_necesarios} alumnos.")
            print(f"   - Cobertura (Incluyendo Sobrecupo 20%): {cupos_logrados} alumnos.")
            print(f"   - 💡 SUGERENCIA: Faltan acomodar {int(cupos_necesarios - cupos_logrados)} alumnos. Requiere contratar Ing. en '{area_nombre}'.")
            print("-" * 60)
            
    if alertas == 0:
        print("✅ EXCELENTE: Usando el margen del 20% de tolerancia en aulas, se cubrió el 100% de la demanda proyectada sin desperdiciar recursos.")
else:
    print("❌ No se encontró solución.")

🎓 CALENDARIO MAESTRO SEMANAL COMPLETADO:


Dia,Lu,Ma,Mi,Ju,Vi
Hora,,,,,
07:00,📚 IS-601 - Base de Datos II (S-1) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 111 (30 norm / 36 ext),📚 IS-601 - Base de Datos II (S-1) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 111 (30 norm / 36 ext),📚 IS-601 - Base de Datos II (S-1) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 111 (30 norm / 36 ext),📚 IS-601 - Base de Datos II (S-1) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 111 (30 norm / 36 ext) 📚 IS-611 - Redes de Datos II (S-1) 👨‍🏫 Ing. René Velasquez 🏫 Virtual (60 norm / 60 ext),📚 IS-611 - Redes de Datos II (S-1) 👨‍🏫 Ing. René Velasquez 🏫 Virtual (60 norm / 60 ext)
08:00,📚 IS-902 - Industria del Software (S-1) 👨‍🏫 Ing. José Marden Argueta 🏫 Aula 111 (30 norm / 36 ext) 📚 ISC-204 - Paradigmas de Programación (S-1) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 323 (30 norm / 36 ext),📚 IS-902 - Industria del Software (S-1) 👨‍🏫 Ing. José Marden Argueta 🏫 Aula 111 (30 norm / 36 ext) 📚 ISC-204 - Paradigmas de Programación (S-1) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 323 (30 norm / 36 ext),📚 IS-902 - Industria del Software (S-1) 👨‍🏫 Ing. José Marden Argueta 🏫 Aula 111 (30 norm / 36 ext) 📚 ISC-204 - Paradigmas de Programación (S-1) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 323 (30 norm / 36 ext),📚 IS-902 - Industria del Software (S-1) 👨‍🏫 Ing. José Marden Argueta 🏫 Aula 111 (30 norm / 36 ext) 📚 ISC-204 - Paradigmas de Programación (S-1) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 323 (30 norm / 36 ext) 📚 IS-611 - Redes de Datos II (S-1) 👨‍🏫 Ing. René Velasquez 🏫 Virtual (60 norm / 60 ext),📚 IS-611 - Redes de Datos II (S-1) 👨‍🏫 Ing. René Velasquez 🏫 Virtual (60 norm / 60 ext)
09:00,📚 ISC-333 - Sistemas Operativos I (S-1) 👨‍🏫 Ing. Edis Francisco Romero 🏫 Aula 111 (30 norm / 36 ext) 📚 ISC-409 - Calidad de Software (S-1) 👨‍🏫 Ing. Elias Emilio Flores 🏫 Aula 323 (30 norm / 36 ext),📚 ISC-333 - Sistemas Operativos I (S-1) 👨‍🏫 Ing. Edis Francisco Romero 🏫 Aula 111 (30 norm / 36 ext) 📚 ISC-409 - Calidad de Software (S-1) 👨‍🏫 Ing. Elias Emilio Flores 🏫 Aula 323 (30 norm / 36 ext),📚 ISC-333 - Sistemas Operativos I (S-1) 👨‍🏫 Ing. Edis Francisco Romero 🏫 Aula 111 (30 norm / 36 ext) 📚 ISC-409 - Calidad de Software (S-1) 👨‍🏫 Ing. Elias Emilio Flores 🏫 Aula 323 (30 norm / 36 ext),📚 ISC-333 - Sistemas Operativos I (S-1) 👨‍🏫 Ing. Edis Francisco Romero 🏫 Aula 111 (30 norm / 36 ext) 📚 ISC-409 - Calidad de Software (S-1) 👨‍🏫 Ing. Elias Emilio Flores 🏫 Aula 323 (30 norm / 36 ext),---
10:00,📚 IS-912 - Sistemas Expertos (S-1) 👨‍🏫 Ing. Asalia Alejandra Zavala 🏫 Virtual (60 norm / 60 ext) 📚 IS-115 - Seminario de Investigación (S-1) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 323 (30 norm / 36 ext) 📚 ISC-334 - Sistemas Operativos II (S-1) 👨‍🏫 Ing. Elias Emilio Flores 🏫 Aula 111 (30 norm / 36 ext),📚 IS-912 - Sistemas Expertos (S-1) 👨‍🏫 Ing. Asalia Alejandra Zavala 🏫 Virtual (60 norm / 60 ext) 📚 IS-115 - Seminario de Investigación (S-1) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 323 (30 norm / 36 ext) 📚 ISC-334 - Sistemas Operativos II (S-1) 👨‍🏫 Ing. Elias Emilio Flores 🏫 Aula 111 (30 norm / 36 ext),📚 IS-912 - Sistemas Expertos (S-1) 👨‍🏫 Ing. Asalia Alejandra Zavala 🏫 Virtual (60 norm / 60 ext) 📚 IS-115 - Seminario de Investigación (S-1) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 323 (30 norm / 36 ext) 📚 ISC-334 - Sistemas Operativos II (S-1) 👨‍🏫 Ing. Elias Emilio Flores 🏫 Aula 111 (30 norm / 36 ext),📚 IS-912 - Sistemas Expertos (S-1) 👨‍🏫 Ing. Asalia Alejandra Zavala 🏫 Virtual (60 norm / 60 ext) 📚 IS-115 - Seminario de Investigación (S-1) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Aula 323 (30 norm / 36 ext) 📚 ISC-334 - Sistemas Operativos II (S-1) 👨‍🏫 Ing. Elias Emilio Flores 🏫 Aula 111 (30 norm / 36 ext) 📚 ISC-331 - Redes de Datos I (S-1) 👨‍🏫 Ing. René Velasquez 🏫 Virtual (60 norm / 60 ext),📚 ISC-331 - Redes de Datos I (S-1) 👨‍🏫 Ing. René Velasquez 🏫 Virtual (60 norm / 60 ext)
11:00,📚 IS-913 - Diseño de Compiladores (S-1) 👨‍🏫 Ing. Oscar Guillermo Hernández 🏫 Virtual (60 norm / 60 ext) 📚 IS-802 - Ingeniería del Software (S-1) 👨‍🏫 Ing. José Ma


📊 REPORTE DE GESTIÓN (DÉFICIT FÍSICO Y TOLERANCIAS)
⚠️ DÉFICIT EN: IS-911 - Microprocesadores
   - Demanda Proyectada: 10 alumnos.
   - Cobertura (Incluyendo Sobrecupo 20%): 0 alumnos.
   - 💡 SUGERENCIA: Faltan acomodar 10 alumnos. Requiere contratar Ing. en 'Arquitectura y Hardware'.
------------------------------------------------------------
⚠️ DÉFICIT EN: IS-811 - Seguridad Informática
   - Demanda Proyectada: 7 alumnos.
   - Cobertura (Incluyendo Sobrecupo 20%): 0 alumnos.
   - 💡 SUGERENCIA: Faltan acomodar 7 alumnos. Requiere contratar Ing. en 'Auditoria Informatica'.
------------------------------------------------------------
⚠️ DÉFICIT EN: IS-701 - Inteligencia Artificial
   - Demanda Proyectada: 6 alumnos.
   - Cobertura (Incluyendo Sobrecupo 20%): 0 alumnos.
   - 💡 SUGERENCIA: Faltan acomodar 6 alumnos. Requiere contratar Ing. en 'Inteligencia Artificial'.
------------------------------------------------------------
⚠️ DÉFICIT EN: IS-602 - Sistema de Información
   - Demand